In [3]:
import os
import re
import numpy as np
import pandas as pd
import cv2
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.color import rgb2gray
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1).to(device)

feature_extractor = torch.nn.Sequential(
    mobilenet.features,
    mobilenet.avgpool,
    torch.nn.Flatten()
)
feature_extractor.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

def build_all_feature_names():
    feat_names = []
    for ch in ["R", "G", "B"]:
        for bin_idx in range(32):
            feat_names.append(f"RGB_{ch}_hist_bin{bin_idx}")
    for ch in ["H", "S", "V"]:
        for bin_idx in range(32):
            feat_names.append(f"HSV_{ch}_hist_bin{bin_idx}")
    stats_type = ["mean", "std", "skewness", "kurtosis"]
    for ch in ["R", "G", "B"]:
        for st in stats_type:
            feat_names.append(f"RGB_{ch}_{st}")
    glcm_props = ["contrast", "dissimilarity", "homogeneity", "energy"]
    angles = ["0deg", "45deg", "90deg", "135deg"]
    for prop in glcm_props:
        for ang in angles:
            feat_names.append(f"GLCM_{prop}_{ang}")
    for bin_idx in range(32):
        feat_names.append(f"LBP_uniform_bin{bin_idx}")
    for bin_idx in range(100):
        feat_names.append(f"Gray_hist_bin{bin_idx}")
    for idx in range(960):
        feat_names.append(f"MobileNetV3Large_deep_{idx:04d}")
    assert len(feat_names) == 1312
    return feat_names

all_feature_names = build_all_feature_names()

def extract_mobilenet_large_features(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read image file: {image_path}")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = rgb2gray(img_rgb)
    img_gray_8bit = (img_gray * 255).astype(np.uint8)
    features = []
    for channel in range(3):
        hist = cv2.calcHist([img_rgb], [channel], None, [32], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        features.extend(hist)
    img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    for channel in range(3):
        hist = cv2.calcHist([img_hsv], [channel], None, [32], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        features.extend(hist)
    for channel in range(3):
        channel_data = img_rgb[:, :, channel].flatten()
        mean_val = np.mean(channel_data)
        std_val = np.std(channel_data) + 1e-8
        skew = np.mean((channel_data - mean_val)**3) / (std_val**3)
        kurt = np.mean((channel_data - mean_val)**4) / (std_val**4)
        features.extend([mean_val, std_val, skew, kurt])
    glcm = graycomatrix(img_gray_8bit,distances=[1],angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],levels=256,symmetric=True,normed=True)
    for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy']:
        features.extend(graycoprops(glcm, prop).flatten())
    lbp = local_binary_pattern(img_gray_8bit,P=8,R=1,method='uniform')
    lbp_hist, _ = np.histogram(lbp, bins=32, range=(0, 32), density=True)
    features.extend(lbp_hist)
    gray_hist, _ = np.histogram(img_gray_8bit, bins=100, range=(0, 256), density=True)
    features.extend(gray_hist)
    pil_img = Image.fromarray(img_rgb)
    img_tensor = transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        deep_features = feature_extractor(img_tensor)
        deep_features = deep_features.squeeze().cpu().numpy()
    features.extend(deep_features)
    return np.array(features, dtype=np.float32)

if __name__ == "__main__":
    IMAGE_DIR = r"E:\TSG\feature boost\Must\2"
    FEATURES_OUTPUT = "image_features_mobilenet_large-1.csv"
    INDEX_MAP_OUTPUT = "image_index_mapping_large-1.csv"
    FEATURE_NAME_OUTPUT = "feature_names.csv"
    print(f"Starting image feature extraction from path: {IMAGE_DIR}")
    print("Model: MobileNetV3-Large, total feature dimension: 1312")
    print("-" * 60)
    pd.DataFrame({"feature_name": all_feature_names}).to_csv(FEATURE_NAME_OUTPUT, index=False, encoding="utf-8-sig")
    all_files = os.listdir(IMAGE_DIR)
    bmp_files = [f for f in all_files if f.lower().endswith('.bmp')]
    def extract_number(filename):
        match = re.search(r'(\d+)', filename)
        return int(match.group(1)) if match else float('inf')
    bmp_files_sorted = sorted(bmp_files, key=extract_number)
    print(f"Found {len(bmp_files_sorted)} BMP files")
    print(f"First 5 files: {bmp_files_sorted[:5]}")
    print(f"Last 5 files: {bmp_files_sorted[-5:]}")
    print("-" * 60)
    all_features = []
    index_mapping = []
    success_count = 0
    failed_files = []
    for i, filename in enumerate(bmp_files_sorted):
        file_path = os.path.join(IMAGE_DIR, filename)
        try:
            features = extract_mobilenet_large_features(file_path)
            if len(features) != 1312:
                raise ValueError(f"Invalid feature dimension: {len(features)} (expected 1312)")
            all_features.append(features)
            index_mapping.append({"original_index": i+1,"filename": filename,"full_path": file_path})
            success_count += 1
        except Exception as e:
            failed_files.append(filename)
    all_features = np.array(all_features)
    print("-" * 60)
    print(f"Processing completed! Success: {success_count}, Failed: {len(failed_files)}")
    if success_count > 0:
        print(f"Feature matrix shape: {all_features.shape}")
        pd.DataFrame(all_features).to_csv(FEATURES_OUTPUT, header=False, index=False)
        print(f"Features saved to: {FEATURES_OUTPUT}")
        pd.DataFrame(index_mapping).to_csv(INDEX_MAP_OUTPUT, index=False, encoding='utf-8-sig')
        print(f"Index mapping saved to: {INDEX_MAP_OUTPUT}")
    print("-" * 60)
    print("Feature extraction completed! Next step:")

Starting image feature extraction from path: E:\TSG\feature boost\Must\2
Model: MobileNetV3-Large, total feature dimension: 1312
------------------------------------------------------------
Found 575 BMP files
First 5 files: ['1.bmp', '2.bmp', '3.bmp', '4.bmp', '5.bmp']
Last 5 files: ['571.bmp', '572.bmp', '573.bmp', '574.bmp', '575.bmp']
------------------------------------------------------------
------------------------------------------------------------
Processing completed! Success: 575, Failed: 0
Feature matrix shape: (575, 1312)
Features saved to: image_features_mobilenet_large-1.csv
Index mapping saved to: image_index_mapping_large-1.csv
------------------------------------------------------------
Feature extraction completed! Next step:
